### Project 1 - House Price

You are provided with a dataset containing information about houses and **prices**. The dataset is already split into `train.csv` and `test.csv`. The `data_description.txt` file contains descriptions of the columns.

**Goal:** Build models to predict house prices.

Please include detailed explanations of the following steps:

1. Data cleaning and preprocessing.

2. Model training and validation.

3. Model comparison based on regression metrics.

**Note:** You are **encouraged** to look for other machine learning algorithms online (beyond the course material), but you must study and understand these algorithms. You must not remove any rows from the `test.csv` file.

## Data Cleaning and Preprocessing

In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler

# Load initial datasets
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# Remove extreme outliers from training data to improve model accuracy
train = train.drop(train[(train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)].index)

# Create a combined square footage feature for both datasets
train['TotalSF'] = train['TotalBsmtSF'].fillna(0) + train['1stFlrSF'].fillna(0) + train['2ndFlrSF'].fillna(0)
test['TotalSF'] = test['TotalBsmtSF'].fillna(0) + test['1stFlrSF'].fillna(0) + test['2ndFlrSF'].fillna(0)

# Apply log transformation to normalize the target variable distribution
train['SalePrice_Log'] = np.log1p(train['SalePrice'])

def clean_and_encode(df):
    df_clean = df.copy()
    
    cols_with_na_meaning = [
        'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 
        'BsmtFinType2', 'FireplaceQu', 'GarageType', 'GarageFinish', 
        'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature', 'MasVnrType'
    ]
    
    # Restore intentional NA categories defined in the data dictionary
    for col in cols_with_na_meaning:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].fillna('None')
            
    # Impute specific numerical gaps with logical defaults
    if 'MasVnrArea' in df_clean.columns:
        df_clean['MasVnrArea'] = df_clean['MasVnrArea'].fillna(0.0)
    if 'LotFrontage' in df_clean.columns:
        df_clean['LotFrontage'] = df_clean['LotFrontage'].fillna(df_clean['LotFrontage'].median())
    if 'GarageYrBlt' in df_clean.columns:
        df_clean['GarageYrBlt'] = df_clean['GarageYrBlt'].fillna(0)
        
    # Catch-all imputation for any remaining missing values to prevent dropping rows
    for col in df_clean.columns:
        if df_clean[col].isnull().sum() > 0:
            if pd.api.types.is_numeric_dtype(df_clean[col]):
                df_clean[col] = df_clean[col].fillna(df_clean[col].median())
            else:
                df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

    # Convert categorical text into binary indicator columns
    df_encoded = pd.get_dummies(df_clean, dtype=int)    
    return df_encoded

# Process both datasets through the cleaning pipeline
train_encoded = clean_and_encode(train)
test_encoded = clean_and_encode(test)

# Align columns to ensure the test set matches the train set feature structure exactly
train_aligned, test_aligned = train_encoded.align(test_encoded, join='left', axis=1, fill_value=0)

# Remove target variables from the test set to prevent data leakage
if 'SalePrice' in test_aligned.columns:
    test_aligned = test_aligned.drop(columns=['SalePrice'])
if 'SalePrice_Log' in test_aligned.columns:
    test_aligned = test_aligned.drop(columns=['SalePrice_Log'])

# Isolate target and predictors before scaling
y_train = train_aligned['SalePrice_Log']
X_train = train_aligned.drop(columns=['SalePrice', 'SalePrice_Log'])
X_test = test_aligned

# Scale numerical features to reduce the impact of remaining outliers
scaler = RobustScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

# Final verification prints
print(f"Final X_train shape: {X_train_scaled.shape}")
print(f"Final y_train shape: {y_train.shape}")
print(f"Final X_test shape: {X_test_scaled.shape}")
print(f"\nMissing values in X_train: {X_train_scaled.isnull().sum().sum()}")
print(f"Missing values in X_test: {X_test_scaled.isnull().sum().sum()}")

Final X_train shape: (1312, 300)
Final y_train shape: (1312,)
Final X_test shape: (146, 300)

Missing values in X_train: 0
Missing values in X_test: 0


# Model Training and Validation

# Model Comparison based on Regression Metrics